# Evaluation Notebook

This file performs evaluation of all algorithms/variants, stores results in a Zarr file, and generates plots.

## Configuration

In [1]:
# Set the output file automatically.
from datetime import datetime
datetime_str = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = "./results/patrolling_comms_" + datetime_str + ".zarr"


# output_file = "/data/anthony/dev/qmas_task_allocation/scripts/results/patrolling_comms_20260425_141122.zarr"

## Setup

In [2]:
%reload_ext autoreload
%autoreload 2
from onpolicy.scripts.eval.eval_pettingzoo import main

import os
import yaml
import time
from copy import deepcopy
from multiprocessing import Process
os.environ["WANDB__SERVICE_WAIT"] = "300"

In [3]:
def convert_args(argsdict):
    # Convert arguments to the appropriate format.
    arglist = []
    def add_arg(arglist, key, value):
        if value == None:
            return
        if type(value) == bool:
            if value:
                arglist.append(f"--{key}")
            else:
                arglist.append(f"--no-{key}")
        elif type(value) == list and len(value) > 0:
            arglist.append(f"--{key}")
            arglist.extend([str(v) for v in value])
        else:
            arglist.append(f"--{key}")
            arglist.append(str(value))
    for key, value in argsdict.items():
        if type(value) == dict and "value" in value and not key.startswith("_"):
            add_arg(arglist, key, value["value"])
        else:
            add_arg(arglist, key, value)
    print("Converted arguments:", arglist)
    return arglist


def wrap_arg_val(val):
    return {"value": val}


def load_args(model_dir):
    # Load arguments from the config file.
    config_file = os.path.join(model_dir, "config.yaml")
    args = yaml.load(open(config_file), Loader=yaml.FullLoader)

    # Set required eval-specific arguments. Do not change these!
    args["use_wandb"] = wrap_arg_val(False)
    args["use_render"] = wrap_arg_val(False)
    args["use_eval"] = wrap_arg_val(True)
    args["model_dir"] = wrap_arg_val(model_dir)
    args["eval_output_file"] = wrap_arg_val(output_file)


    # Set default arguments for evaluation. Feel free to change these!
    args["cuda"] = wrap_arg_val(True)
    args["cuda_idx"] = wrap_arg_val(1)
    args["cuda_idx_predictor"] = args["cuda_idx"]
    args["eval_episodes"] = wrap_arg_val(3)
    args["episode_length"] = wrap_arg_val(200)
    args["seed"] = wrap_arg_val(1)

    args["observation_mask"] = wrap_arg_val(True)

    args["diffusion_autoregression_steps"] = wrap_arg_val(1)

    args["graph_name"] = wrap_arg_val("cumberland")
    args["graph_file"] = wrap_arg_val("../patrolling_zoo/graphs/cumberland.graph")
    args["graph_random"] = wrap_arg_val(False)

    return args

# Evaluation

In [4]:
# obs_radii = [0.0, 50.0, 100.0, 150.0, 200.0, 250.0, 300.0]
obs_radii = [50.0, 100.0, 150.0, 200.0]
comms_probability = 0.1

# Check if the output file already exists. If so, skip this cell.
if os.path.exists(output_file):
    print("Output file already exists. Skipping evaluation.")

else:
    processes = []

    def start_process(args):
        p = Process(target=main, args=[convert_args(args)])
        processes.append(p)

    for obs_radius in obs_radii:
        args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/rmappo/patrolling-random-draft1/wandb/run-20260425_154750-gehotzfo/files")
        args["eval_series"] = args["experiment_name"]
        # args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/rmappo/patrolling-random-pred-draft1/wandb/run-20260425_154745-5z2rswp2/files")
        # args["eval_series"] = wrap_arg_val(f"no-prediction-{args['experiment_name']['value']}")
        args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}")
        args["observation_radius"] = wrap_arg_val(obs_radius)
        args["communication_probability"] = wrap_arg_val(comms_probability)
        # args["prediction_disable"] = wrap_arg_val(True)
        start_process(args)

        args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/rmappo/patrolling-random-pred-draft1/wandb/run-20260425_154745-5z2rswp2/files")
        args["eval_series"] = args["experiment_name"]
        args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}")
        args["observation_radius"] = wrap_arg_val(obs_radius)
        args["communication_probability"] = wrap_arg_val(comms_probability)
        start_process(args)

        args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/rmappo/patrolling-random-pred-uq-draft1/wandb/run-20260425_154740-nfpyui2q/files")
        args["eval_series"] = args["experiment_name"]
        args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}")
        args["observation_radius"] = wrap_arg_val(obs_radius)
        args["communication_probability"] = wrap_arg_val(comms_probability)
        start_process(args)

        args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/rmappo/patrolling-random-comms-pred-draft1/wandb/run-20260425_154736-8meq3s4s/files")
        args["eval_series"] = args["experiment_name"]
        args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}")
        args["observation_radius"] = wrap_arg_val(obs_radius)
        args["communication_probability"] = wrap_arg_val(comms_probability)
        start_process(args)

        args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/rmappo/patrolling-random-comms-pred-uq-draft1/wandb/run-20260425_154718-fric9zzy/files")
        args["eval_series"] = args["experiment_name"]
        args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}")
        args["observation_radius"] = wrap_arg_val(obs_radius)
        args["communication_probability"] = wrap_arg_val(comms_probability)
        start_process(args)


    # JOIN ALL PROCESSES
    # Up to MAX_PROCESSES processes can run simultaneously.
    MAX_PROCESSES = 25
    running_processes = []
    time_start = time.time()
    failed_processes = []
    
    while len(processes) > 0 or len(running_processes) > 0:
        # Start new processes if we have capacity.
        while len(running_processes) < MAX_PROCESSES and len(processes) > 0:
            p = processes.pop(0)
            p.start()
            running_processes.append(p)
        
        # Check for finished processes.
        for p in running_processes:
            if not p.is_alive():
                p.join()
                if p.exitcode != 0:
                    failed_processes.append(p)
        running_processes = [p for p in running_processes if p.is_alive()]
    
    time_end = time.time()
    print(f"Total evaluation time: {time_end - time_start:.2f} seconds")
    
    if failed_processes:
        raise RuntimeError(f"{len(failed_processes)} subprocess(es) failed with non-zero exit codes")
    
    print("All processes finished successfully!")

Converted arguments: ['--_wandb', "{'value': {'cli_version': '0.19.11', 'm': [], 'python_version': '3.10.16', 't': {'1': [1, 55, 77, 105], '2': [1, 9, 41, 49, 55, 63, 74, 77, 103, 105], '3': [2, 13, 16, 23, 55, 61], '4': '3.10.16', '5': '0.19.11', '8': [5], '12': '0.19.11', '13': 'linux-x86_64'}}}", '--accept_ratio', '0.5', '--action_full_max_nodes', '40', '--action_method', 'neighbors', '--action_neighbors_max_degree', '10', '--agent_speed', '40', '--algorithm_class', 'qmas.variant_two_modules.new.algorithm.QmasAlgorithm', '--algorithm_name', 'rmappo', '--alpha', '1', '--attrition_fixed_times', '[]', '--attrition_method', 'none', '--attrition_min_agents', '2', '--attrition_random_probability', '0', '--beta', '0.1', '--broadcast_uncertainty_threshold', '0.5', '--clip_param', '0.2', '--communication_model', 'bernoulli', '--communication_probability', '0.1', '--critic_hidden_size', '64', '--critic_lr', '0.0005', '--cuda', '--cuda_deterministic', '--cuda_idx', '1', '--cuda_idx_predictor',

/home/anthony/data/dev/qmas_task_allocation/onpolicy/scripts/train/train_pettingzoo.py:158: UserWarning: Unknown arguments: ['--_wandb', "{'value': {'cli_version': '0.19.11', 'm': [], 'python_version': '3.10.16', 't': {'1': [1, 55, 77, 105], '2': [1, 9, 41, 49, 55, 63, 74, 77, 103, 105], '3': [2, 13, 16, 23, 55, 61], '4': '3.10.16', '5': '0.19.11', '8': [5], '12': '0.19.11', '13': 'linux-x86_64'}}}", '--episode_max', '75', '--no-share_reward', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy']
  warnings.warn(f"Unknown arguments: {unknown_args}")
/home/anthony/data/dev/qmas_task_allocation/onpolicy/scripts/train/train_pettingzoo.py:158: UserWarning: Unknown arguments: ['--_wandb', "{'value': {'cli_version': '0.19.11', 'm': [], 'python_version': '3.10.16', 't': {'1': [1, 55, 77, 105], '2': [1, 9, 41, 49, 55, 63, 74, 77, 103, 105], '3': [2, 13, 16, 23, 55, 61], '4': '3.10.16', '5': '0.19.11', '8': [5], '12': '0.19.11', '13': 'linux-x86_64'}}}", '--episode_max', '75',

u are choosing to use rmappo, we set use_recurrent_policy to be Trueu are choosing to use rmappo, we set use_recurrent_policy to be TruePettingzoo environment patrolling_zoo.patrolling_zoo_v0.parallel_env has additional arguments.




/home/anthony/data/dev/qmas_task_allocation/onpolicy/scripts/train/train_pettingzoo.py:158: UserWarning: Unknown arguments: ['--_wandb', "{'value': {'cli_version': '0.19.11', 'm': [], 'python_version': '3.10.16', 't': {'1': [1, 55, 77, 105], '2': [1, 9, 41, 49, 55, 63, 74, 77, 103, 105], '3': [2, 13, 16, 23, 55, 61], '4': '3.10.16', '5': '0.19.11', '8': [5], '12': '0.19.11', '13': 'linux-x86_64'}}}", '--episode_max', '75', '--no-share_reward', '--no-torch_compile', '--no-use_gnn_critic', '--no-use_gnn_mlp_policy']
  warnings.warn(f"Unknown arguments: {unknown_args}")


Pettingzoo environment patrolling_zoo.patrolling_zoo_v0.parallel_env has additional arguments.u are choosing to use rmappo, we set use_recurrent_policy to be True



...

Pettingzoo arguments validated: base
Pettingzoo environment patrolling_zoo.patrolling_zoo_v0.parallel_env has additional arguments validation function.
Pettingzoo arguments validated: patrolling_zoo.patrolling_zoo_v0.parallel_env
PettingzooEnv: Attempting to pass unpacked argparse namespace to environment.
PettingzooEnv: Passing the following arguments to the environment: {'max_cycles': -1, 'num_agents': 4, 'alpha': 1.0, 'beta': 0.1, 'reward_comms_penalty_weight': 0.1, 'graph_file': '../patrolling_zoo/graphs/cumberland.graph', 'graph_random': False, 'graph_random_nodes': 40, 'reward_method_terminal': 'none', 'reward_interval': 1, 'agent_speed': 40.0, 'action_method': 'neighbors_with_comm_boolean', 'observe_method': 'pyg', 'observe_method_global': 'coordinates', 'observe_bitmap_size': 50, 'observation_radius': 100.0, 'attrition_method': 'none', 'attrition_fixed_times': ['[', ']'], 'attrition_random_probability': 0.0, 'attrition_min_agents': 2, 'communication_model': 'none', 'communicati

/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(


Pettingzoo arguments validated: basePettingzoo arguments validated: patrolling_zoo.patrolling_zoo_v0.parallel_env
PettingzooEnv: Attempting to pass unpacked argparse namespace to environment.
Pettingzoo environment patrolling_zoo.patrolling_zoo_v0.parallel_env has additional arguments validation function.

PettingzooEnv: Passing the following arguments to the environment: {'max_cycles': -1, 'num_agents': 4, 'alpha': 1.0, 'beta': 0.1, 'reward_comms_penalty_weight': 0.1, 'graph_file': '../patrolling_zoo/graphs/cumberland.graph', 'graph_random': False, 'graph_random_nodes': 40, 'reward_method_terminal': 'none', 'reward_interval': 1, 'agent_speed': 40.0, 'action_method': 'neighbors_with_comm_boolean', 'observe_method': 'pyg', 'observe_method_global': 'coordinates', 'observe_bitmap_size': 50, 'observation_radius': 200.0, 'attrition_method': 'none', 'attrition_fixed_times': ['[', ']'], 'attrition_random_probability': 0.0, 'attrition_min_agents': 2, 'communication_model': 'none', 'communicati

/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(
/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(
/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(
/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(
/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/torchrl/__init__.py:48: UserWarning: failed to set start method to spawn, and current start method for mp is fork.
  warn(
/home/anthony/miniconda3/envs/qmas/lib/python3.10/site-packages/t

R_Actor: Use GNN: True, Use CNN: False, Use MLP: False, Use RNN: True, Use Encoder: True
State encoder use_cnn: False, input_dim: 129, output_dim: 32
Action output distributions:
  0: Categorical, action_dim: 1, available_action_dim: 10
R_Actor: Use GNN: True, Use CNN: False, Use MLP: False, Use RNN: True, Use Encoder: True
State encoder use_cnn: False, input_dim: 129, output_dim: 32
Action output distributions:
  0: Categorical, action_dim: 1, available_action_dim: 2
  1: Categorical, action_dim: 1, available_action_dim: 10
R_Actor: Use GNN: True, Use CNN: False, Use MLP: False, Use RNN: True, Use Encoder: True
State encoder use_cnn: False, input_dim: 258, output_dim: 32
Action output distributions:
  0: Categorical, action_dim: 1, available_action_dim: 10
R_Actor: Use GNN: True, Use CNN: False, Use MLP: False, Use RNN: True, Use Encoder: True
State encoder use_cnn: False, input_dim: 129, output_dim: 32
Action output distributions:
  0: Categorical, action_dim: 1, available_action_dim

# Analysis

In [5]:
import zarr
output = zarr.open(output_file, mode='r')
output.tree()

Tree(nodes=(Node(disabled=True, name='/', nodes=(Node(disabled=True, name='patrolling-random-comms-pred-draft1…

In [6]:
import plotly.graph_objects as go
import plotly.express as px
import numpy as np

def sorted_experiments(series):
    """Return experiments in a series sorted by observation_radius attribute."""
    return sorted(output[series], key=lambda exp: output[series][exp].attrs['observation_radius'])

def get_x_radii(series):
    """Read the observation_radius attribute for each experiment, sorted ascending."""
    return [output[series][exp].attrs['observation_radius'] for exp in sorted_experiments(series)]

# Data for plotting
categories = {
    'prediction_error_mean': {
        'title': 'Prediction Error',
        'ylabel': 'Prediction Error (units distance)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_error_mean']) for exp in sorted_experiments(series)]
                for series in output
        }
    },
    'prediction_trajectory_error_mean': {
        'title': 'Prediction Trajectory Error',
        'ylabel': 'Prediction Trajectory Error (units distance)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_trajectory_error_mean']) for exp in sorted_experiments(series)]
                for series in output
        }
    },
    'avg_idleness': {
        'title': 'Time-Averaged Avg. Idleness',
        'ylabel': 'Avg. Node Idleness (s)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['avg_idleness']) for exp in sorted_experiments(series)]
                for series in output
        }
    },
    'avg_idleness_final': {
        'title': 'Final Avg. Idleness',
        'ylabel': 'Avg. Node Idleness (s)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series) for series in output},
        'y': {
            series: [np.mean(output[series][exp]['avg_idleness'][:, -1]) for exp in sorted_experiments(series)]
                for series in output
        }
    },
    'avg_idleness_over_time': {
        'title': 'Avg. Idleness vs. Time',
        'ylabel': 'Avg. Node Idleness (s)',
        'xlabel': 'Time Steps',
        # One trace per experiment: average over episodes, keep time dimension.
        'y': {
            series: {
                exp: np.mean(output[series][exp]['avg_idleness'], axis=0)
                for exp in sorted_experiments(series)
            }
            for series in output
        }
    },
    'worst_idleness': {
        'title': 'Time-Averaged Worst Idleness',
        'ylabel': 'Worst Node Idleness (s)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['worst_idleness']) for exp in sorted_experiments(series)]
                for series in output
        }
    },
    'worst_idleness_final': {
        'title': 'Final Worst Idleness',
        'ylabel': 'Worst Node Idleness (s)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series) for series in output},
        'y': {
            series: [np.mean(output[series][exp]['worst_idleness'][:, -1]) for exp in sorted_experiments(series)]
                for series in output
        }
    },
    'worst_idleness_over_time': {
        'title': 'Worst Idleness vs. Time',
        'ylabel': 'Worst Node Idleness (s)',
        'xlabel': 'Time Steps',
        # One trace per experiment: average over episodes, keep time dimension.
        'y': {
            series: {
                exp: np.mean(output[series][exp]['worst_idleness'], axis=0)
                for exp in sorted_experiments(series)
            }
            for series in output
        }
    },
    'stddev_idleness': {
        'title': 'Time-Averaged Std. Dev. Idleness',
        'ylabel': 'Std. Dev. Node Idleness (s)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['stddev_idleness']) for exp in sorted_experiments(series)]
                for series in output
        }
    },
    'stddev_idleness_over_time': {
        'title': 'Std. Dev. Idleness vs. Time',
        'ylabel': 'Std. Dev. Node Idleness (s)',
        'xlabel': 'Time Steps',
        # One trace per experiment: average over episodes, keep time dimension.
        'y': {
            series: {
                exp: np.mean(output[series][exp]['stddev_idleness'], axis=0)
                for exp in sorted_experiments(series)
            }
            for series in output
        }
    },
    'communication_requests': {
        'title': 'Communication Requests vs. Observation Radius',
        'ylabel': 'Communication Requests',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series) for series in output},
        'y': {
            series: [np.mean(output[series][exp]['communication']['requests_made']) for exp in sorted_experiments(series)]
                for series in output
        }
    },
    'communication_requests_time': {
        'title': 'Cumulative Communication Requests vs. Time',
        'ylabel': 'Communication Requests',
        'xlabel': 'Time Steps',
        'y': {
            series: {
                exp: np.cumsum(np.mean(output[series][exp]['communication']['requests_made'], axis=0))
                for exp in sorted_experiments(series)
            }
            for series in output
        }
    },
    'prediction_uncertainty_mean': {
        'title': 'Prediction Uncertainty',
        'ylabel': 'Prediction Uncertainty (units distance)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_uncertainty_mean']) for exp in sorted_experiments(series)]
                for series in output
        }
    },
    'prediction_uncertainty_gap_mean': {
        'title': 'Prediction Uncertainty Gap',
        'ylabel': 'Prediction Uncertainty Gap (units distance)',
        'xlabel': 'Observation Radius',
        'x': {series: get_x_radii(series) for series in output},
        # Average over all episodes and all timesteps.
        'y': {
            series: [np.mean(output[series][exp]['prediction_uncertainty_gap_mean']) for exp in sorted_experiments(series)]
                for series in output
        }
    },
}

# Miscelaneous statistics.
timesteps = [np.max([np.max(output[series][experiment]['deltaSteps'].shape[1]) for experiment in output[series]]) for series in output]

for category in categories:
    fig = go.Figure()
    x = categories[category].get('x', None)
    y_data = categories[category]['y']
    show_legend = True

    # Detect nested dict: {series: {experiment: array}} — data with a time dimension.
    is_time_series = (
        isinstance(y_data, dict)
        and len(y_data) > 0
        and isinstance(next(iter(y_data.values())), dict)
    )

    if is_time_series:
        series_names = list(y_data.keys())
        series_names = [name.split(" ")[0] for name in series_names]
        series_names = set(series_names)
        series_colors = {name: color for name, color in zip(series_names, px.colors.qualitative.Plotly)}
        for series, tests in y_data.items():
            series_name = series.split(" ")[0]
            for test_name, values in tests.items():
                x_local = np.arange(1, len(values) + 1)
                fig.add_trace(go.Scatter(
                    x=x_local,
                    y=values,
                    mode='lines',
                    name=f"{series} | {test_name}",
                    hovertemplate=f'{series} | {test_name}' + ': %{y:.4f}<extra></extra>',
                    line={
                        "color": series_colors.get(series_name, 'black'),
                    },
                    legendgroup=series_name,
                    # legendgrouptitle={series_name: series_name},
                ))
    elif isinstance(y_data, dict):
        for series, values in y_data.items():
            # x may be a per-series dict (keyed by series name) or a shared list.
            x_local = x[series] if isinstance(x, dict) else x
            if x_local is None:
                x_local = np.linspace(0, 1, len(values))
            if len(values) == 1:
                # Plot a bar instead of a point if there's only one value, to make it more visible.
                fig.add_trace(go.Bar(
                    x=[series],
                    y=values,
                    name=series,
                    hovertemplate=f'{series}' + ': %{y:.4f}<extra></extra>',
                ))
                show_legend = False  # Legend is not needed if we're labeling bars directly.
            else:
                fig.add_trace(go.Scatter(
                    x=x_local,
                    y=values,
                    mode='lines+markers',
                    name=series,
                    hovertemplate='%{x:.2f}: %{y:.4f}<extra></extra>'
                ))
    else:
        values = y_data
        x_local = x
        if x_local is None:
            x_local = np.linspace(0, 1, len(values))
        fig.add_trace(go.Scatter(
            x=x_local,
            y=values,
            mode='lines+markers',
            hovertemplate='%{x:.2f}: %{y:.4f}<extra></extra>'
        ))

    # Collect the union of all x tick values across series, if x is defined.
    if x is not None and not is_time_series:
        all_x_vals = sorted(set(v for vals in (x.values() if isinstance(x, dict) else [x]) for v in vals))
        xaxis_opts = dict(tickvals=all_x_vals, ticktext=[str(v) for v in all_x_vals])
    else:
        xaxis_opts = {}

    title = categories[category]['title']
    xlabel = 'Scenario' if "xlabel" not in categories[category] else categories[category]['xlabel']
    ylabel = f"Timesteps out of {timesteps}" if "ylabel" not in categories[category] else categories[category]['ylabel']
    
    fig.update_layout(
        # title=f"{title}: avg. over {args['eval_episodes']['value']} episodes, seed {args['seed']['value']}",
        title=f"{title}",
        xaxis_title=xlabel,
        yaxis_title=ylabel,
        xaxis=xaxis_opts,
        hovermode='closest',
        showlegend=show_legend,
        autosize=True,
        height=600,
        margin=dict(l=60, r=20, t=60, b=120),
        legend=dict(
            orientation='h',
            yanchor='top',
            y=-0.2,
            xanchor='center',
            x=0.5
        ),
    )
    
    fig.show()
